In [ ]:
import pandas as pd
data_frame = pd.read_csv('RQ2.csv') #Replace the file for RQ2 output

In [ ]:
# Create a dictionary to store the version numbers for each ProxyAddress
version_dict = {}

# Initialize the version number to 0 for each unique ProxyAddress
data_frame['VersionNumber'] = 0

# Iterate through each row in the DataFrame (in reverse order)
for index, row in data_frame.iloc[::-1].iterrows():
    proxy_address = row['ProxyAddress']
    
    # If the ProxyAddress is already present in the dictionary, increment the version number
    if proxy_address in version_dict:
        version_dict[proxy_address] += 1
    else:
        version_dict[proxy_address] = 1
    
    # Assign the version number to the corresponding row in the DataFrame
    data_frame.at[index, 'VersionNumber'] = version_dict[proxy_address]

# Save the updated DataFrame back to the file
data_frame.to_csv('your_file.csv', index=False)

In [ ]:
import pandas as pd

# Read the data frame with version numbers
df_version = pd.read_csv('your_file.csv')

# Read the data frame with filename and findings
#df_findings = pd.read_csv('VULF.csv') Used in Sampling
df_findings = pd.read_csv('/Users/elyhabaro/Desktop/Results/outputFF.csv')

# Create a mapping dictionary using 'ProxyAddress' as keys and 'implementations' as values
address_to_implementation = dict(zip(df_version['ProxyAddress'], df_version['implementations']))

# Extract the address from the filename and store it in a new 'address' column in df_findings
df_findings['address'] = "0x" + df_findings['filename'].str.extract(r"0x([a-fA-F0-9]{40})")

# Create a new column 'Findings' in df_version and initialize it with 'no source code'
df_version['Findings'] = 'no source code'

# Iterate through each row in df_version
for index, row in df_version.iterrows():
    implementation = row['implementations']
    
    # Check if the implementation address is present in df_findings
    if implementation in df_findings['address'].values:
        # Get the corresponding findings from df_findings
        findings = df_findings[df_findings['address'] == implementation]['findings'].values[0]
        
        # Update the 'Findings' column in df_version with the corresponding findings
        df_version.at[index, 'Findings'] = findings

# Save the updated DataFrame back to the file
df_version.to_csv('DFV.csv', index=False)


In [ ]:
import pandas as pd

# Read the data from the CSV file
df = pd.read_csv('DFV.csv')

# Drop duplicates in the 'implementations' column
df.drop_duplicates(subset='implementations', inplace=True)

# Sort the data frame based on 'ProxyAddress' and 'VersionNumber' in ascending order
df.sort_values(by=['ProxyAddress', 'VersionNumber'], ascending=[True, True], inplace=True)

# Function to process the Findings column
def process_findings(findings):
    # Remove '{' and '}' from the findings
    findings = findings.replace('{', '').replace('}', '')

    # Split the findings by ','
    findings_list = findings.split(',')
    findings_list = [finding.strip() for finding in findings_list]

    return set(findings_list)

# Function to compare findings lists between the versions
def compare_findings(prev_findings, curr_findings):
    added_data = []
    deleted_data = []
    curr_findings_copy = curr_findings.copy()

    for finding in prev_findings:
        if finding in curr_findings_copy:
            curr_findings_copy.remove(finding)
        else:
            deleted_data.append(finding)

    added_data = curr_findings_copy

    return added_data, deleted_data

# Create new columns to store added and deleted data
df['new-Vul'] = ''
df['fix-Vul'] = ''

# Group the data by 'ProxyAddress'
grouped = df.groupby('ProxyAddress')

# Iterate through each group and compare findings for consecutive versions
for name, group in grouped:
    # Convert the Findings column to a list of findings for easier comparison
    findings_list = group['Findings'].tolist()

    # Start comparing from the second version (index 1)
    for i in range(1, len(findings_list)):
        prev_findings = process_findings(findings_list[i-1])
        curr_findings = process_findings(findings_list[i])

        # Find added and deleted data
        added_data, deleted_data = compare_findings(prev_findings, curr_findings)

        # Update 'new-Vul' and 'fix-Vul' columns
        if added_data:
            df.at[group.index[i], 'new-Vul'] = ', '.join(added_data)
        else:
            df.at[group.index[i], 'new-Vul'] = 'no new-vul'

        if deleted_data:
            df.at[group.index[i], 'fix-Vul'] = ', '.join(deleted_data)
        else:
            df.at[group.index[i], 'fix-Vul'] = 'no fixed-vul'

        # Log 'V1' for the first version in each group
        if i == 1:
            df.at[group.index[i - 1], 'new-Vul'] = 'V1'
            df.at[group.index[i - 1], 'fix-Vul'] = 'V1'

# Save the updated DataFrame back to the file
df.to_csv('VV2F.csv', index=False) 



In [ ]:
import pandas as pd

# Assuming you have two DataFrames: df_filename with 'filename' column and df_implementation with 'implementation' and 'proxyAddress' columns
# df_filename = pd.read_csv('VULF.csv')
# df_implementation = pd.read_csv('VV2.csv')
df_filename = pd.read_csv('/Users/elyhabaro/Desktop/Results/outputFF.csv')
df_implementation = pd.read_csv('VV2F.csv')

# Extract the address from the filename and add it as a new column 'address' in df_filename
df_filename['address'] = "0x" + df_filename['filename'].str.extract(r"0x([a-fA-F0-9]{40})")

# Function to find the filename based on the address and implementation
def find_filename(row):
    address = row['implementations']
    matched_filename = df_filename[df_filename['address'] == address]['filename'].values
    if len(matched_filename) > 0:
        return matched_filename[0]
    else:
        return None

# Apply the function to create the new 'matched_filename' column in df_implementation
df_implementation['matched_filename'] = df_implementation.apply(find_filename, axis=1)

# Save the updated DataFrame back to the file
df_implementation.to_csv('your_updated_file.csv', index=False)


In [ ]:
import pandas as pd
import re
import os
def extract_functions_from_file(filepath):
    with open(filepath, 'r') as file:
        content = file.read()

    # Use regex to extract functions from the Solidity file
    function_pattern = re.compile(r'function\s+(.*?)\s*\(')
    functions = function_pattern.findall(content)

    return set(functions)

def compare_function_lists(old_functions, new_functions):
    added_functions = new_functions - old_functions
    deleted_functions = old_functions - new_functions
    return added_functions, deleted_functions

# Assuming you have a DataFrame called df_versions with a column "path" containing file paths
df_versions=pd.read_csv('your_updated_file.csv')
# Create a new column to store the added functions
df_versions['new-feature'] = ''
df_versions['deleted-feature'] = ''
df_versions.sort_values(by=['ProxyAddress', 'VersionNumber'], ascending=[True, True], inplace=True)

# Group the data by 'ProxyAddress'
grouped = df_versions.groupby('ProxyAddress')

# Iterate through each group and compare functions for consecutive versions
for name, group in grouped:
    # Get the file paths for the group
    file_paths = group['matched_filename'].tolist()
    # Extract functions from the files and handle missing or invalid file paths
    functions_list = []
    for filepath in file_paths:
        if pd.notna(filepath) and os.path.exists(filepath):
            functions_list.append(extract_functions_from_file(filepath))
        else:
            functions_list.append(set())

    # Start comparing from the second version (index 1)
    for i in range(1, len(functions_list)):
        prev_functions = functions_list[i-1]
        curr_functions = functions_list[i]

        # Find added and deleted functions
        added_functions, deleted_functions = compare_function_lists(prev_functions, curr_functions)

        # Update 'new-feature' column
        if added_functions:
            df_versions.at[group.index[i], 'new-feature'] = ', '.join(added_functions)
        else:
            df_versions.at[group.index[i], 'new-feature'] = 'no new feature'
            
        # Update 'deleted-feature' column for deleted functions
        if deleted_functions:
            df_versions.at[group.index[i], 'deleted-feature'] = ', '.join(deleted_functions)
        else:
            df_versions.at[group.index[i], 'deleted-feature'] = 'no deleted feature'


        # Log 'V1' for the first version in each group
        if i == 1:
            df_versions.at[group.index[i - 1], 'new-feature'] = 'V1'
            df_versions.at[group.index[i - 1], 'deleted-feature'] = 'V1'
            
# Save the updated DataFrame back to the file
df_versions.to_csv('VVFF.csv', index=False)


In [ ]:
def process_contract_addresses(input_file, output_file, etherscan_api_key):
    # Read input CSV into a Pandas DataFrame
    df = pd.read_csv(input_file)

    # Initialize new columns in the DataFrame to store contract information
    df['gas'] = None
    for index, row in df.iterrows():
        contract_address = row['implementations']
        try:
            gas = get_contract_gas(contract_address, etherscan_api_key)
            df.at[index, 'gas'] = gas
        except Exception as e:
            print(f"Error processing contract address {contract_address}: {e}")
            df.at[index, 'gas'] = "N/A"
    # Save the updated DataFrame to the output CSV file
    df.to_csv(output_file, index=False)

if __name__ == "__main__":
    input_csv_file = "VVFF.csv"  # Replace this with the path to your input CSV file
    output_csv_file = "VFFG1_results.csv"  # Replace this with the desired path for the output CSV file
    etherscan_api_key = ""  # Replace this with your Etherscan API key

    process_contract_addresses(input_csv_file, output_csv_file, etherscan_api_key)
    print("Contract information saved to", output_csv_file)

In [ ]:
# Assuming you already have the DataFrame 'df' with 'gas' column, and 'ProxyAddress' and 'VersionNumber' columns.
df=pd.read_csv('VFFG1_results.csv')
# Initialize a new 'Optimization' column in the DataFrame
df['Optimization'] = None

# Group the data by 'ProxyAddress'
grouped = df.groupby('ProxyAddress')

result_df = pd.DataFrame()

# Iterate through each group and compare gas for consecutive versions
for name, group in grouped:
    group.sort_values(by='VersionNumber', ascending=True, inplace=True)

    # For the first version, set 'Optimization' to 'V1' (no previous version to compare)
    group.loc[group.index[0], 'Optimization'] = 'V1'

    # Compare gas of current version with the previous version
    for i in range(1, len(group)):
        current_gas = group.loc[group.index[i], 'gas']
        previous_gas = group.loc[group.index[i - 1], 'gas']

        # If the gas of the current version is less than the previous version, mark it as 'Optimized'
        if current_gas < previous_gas:
            group.loc[group.index[i], 'Optimization'] = 'Optimized'
        else:
            group.loc[group.index[i], 'Optimization'] = 'Not Optimized'

    # Append the group to the result DataFrame
    result_df = result_df.append(group)

# Sort the DataFrame back to its original order
result_df.sort_index(inplace=True)

# Save the result DataFrame to a new CSV file
output_csv_file = 'output_results1.csv'
result_df.to_csv(output_csv_file, index=False)

In [ ]:
import pandas as pd

# Assuming you have the DataFrame 'df' with 'new-Vul', 'fix-Vul', 'new-feature', 'deleted-feature', and 'Optimization' columns.
df=pd.read_csv('output_results1.csv')

# Create a new "Root-Cause" column and initialize it with empty strings
df['Root-Cause'] = ''

# Step 1: Check fix-Vul column for 'V1' and "no fixed-vul"
fix_vul_condition = (df['fix-Vul'] == 'V1') | (df['fix-Vul'].str.contains('no fixed-vul', case=False, na=False))
df.loc[~fix_vul_condition, 'Root-Cause'] += 'fix-vulnerability,'

# Step 2: Check new-feature column for 'V1' and "no new feature"
new_feature_condition = (df['new-feature'] == 'V1') | (df['new-feature'].str.contains('no new feature', case=False, na=False))
df.loc[~new_feature_condition, 'Root-Cause'] += 'new-feature,'

# Step 3: Check Optimization column for 'Optimized'
optimization_condition = (df['Optimization'] == 'Optimized')
df.loc[optimization_condition, 'Root-Cause'] += 'gas-optimized,'

# Step 4: For every V1 in fix-Vul, write 'V1' in the new column
v1_fix_vul_condition = (df['fix-Vul'] == 'V1')
df.loc[v1_fix_vul_condition, 'Root-Cause'] += 'V1,'

# Remove trailing ',' from the 'Root-Cause' column
df['Root-Cause'] = df['Root-Cause'].str.rstrip(',')

# Replace empty cells with 'others' in the 'Root-Cause' column
df['Root-Cause'].replace('', 'others', inplace=True)

# Save the updated DataFrame to a new CSV file
output_csv_file = 'RQ3_Final.csv'
df.to_csv(output_csv_file, index=False)
